# Mistral chat para calidad del aire

Notebook de referencia para el LLM de la app.

Usa la misma logica que Streamlit:

- lee la API key desde `.confing` (`MISTRAL_API_KEY`);
- construye una tabla actual y una tabla de predicciones bien formateadas;
- genera un prompt para chat con la pregunta del usuario;
- genera otro prompt para el resumen general inicial.

In [ ]:
# Si hace falta instalarlo en un entorno nuevo:
# %pip install mistralai pandas

from pathlib import Path
import sys

import pandas as pd

In [ ]:
def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "app" / "air_quality_llm.py").exists():
            return candidate
    raise FileNotFoundError("No se ha encontrado app/air_quality_llm.py desde el directorio actual.")


ROOT = find_project_root()
sys.path.insert(0, str(ROOT / "app"))

from air_quality_llm import (  # noqa: E402
    CURRENT_PATH,
    PREDICTIONS_PATH,
    ask_air_quality,
    build_dynamic_context,
    build_general_summary_prompt,
    build_user_prompt,
    format_markdown_table,
    load_air_table,
    summarize_air_quality,
)

ROOT

In [ ]:
current_df = load_air_table(CURRENT_PATH)
predictions_df = load_air_table(PREDICTIONS_PATH)

display(current_df.head())
display(predictions_df.head())

## Tablas formateadas para el prompt

Cada celda incluye `valor ug/m3 (calidad)`. Los datos ausentes aparecen como `ND`.

In [ ]:
print(format_markdown_table(current_df, "TABLA ACTUAL SCRAPEADA"))

In [ ]:
print(format_markdown_table(predictions_df, "TABLA DE PREDICCIONES"))

## Contexto dinamico completo

Este bloque es el que se anade en cada llamada: tabla actual + tabla de predicciones.

In [ ]:
print(build_dynamic_context())

## Prompt para chat

In [ ]:
example_question = "Que zona esta peor ahora para NO2 y que predice el modelo?"
print(build_user_prompt(example_question)[:3500])

## Prompt para resumen general inicial

In [ ]:
print(build_general_summary_prompt()[:3500])

## Llamadas reales a Mistral

Estan comentadas para no gastar llamadas por accidente.

In [ ]:
# Chat:
# answer = ask_air_quality("Que contaminante preocupa mas ahora mismo en Valencia?")
# print(answer)

# Resumen general inicial:
# summary = summarize_air_quality()
# print(summary)